<a href="https://colab.research.google.com/github/webtech677/AI-chatbot/blob/main/Trash_AI_RAG_Chatbot_Complete_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trash AI — Image Detection + RAG Chatbot (Google Colab)
**ต่อยอดจาก `trash_detection_full-1(2).ipynb` ของเพื่อน** — เก็บแนวทาง YOLO11s classification + YOLO-World + OpenCLIP ของเดิม และเพิ่ม ChromaDB, Gemini, Chat Memory, Gradio

**วิธีใช้:** `Runtime → Change runtime type → T4 GPU` ถ้ามี → `Runtime → Run all` → รอฝึกโมเดล (หากยังไม่มี `best.pt`) → เพิ่ม `GEMINI_API_KEY` ใน Colab Secrets → เปิดลิงก์ Gradio ที่ปรากฏท้าย Notebook

- **ไม่จำเป็นต้องฝึกใหม่** ถ้ามีไฟล์น้ำหนักที่ `/content/best.pt` หรือ `/content/drive/MyDrive/trash_ai/best.pt`
- **ถ้าไม่มีน้ำหนัก** Notebook นี้จะดาวน์โหลด Dataset และฝึกใหม่ (อาจใช้เวลานาน)
- เนื้อหาความรู้ 10 ประเภทด้านล่างเป็น **ข้อมูลตัวอย่างสำหรับเดโม** ไม่ใช่ฐานอ้างอิงทางการ; ตรวจทาน/เพิ่มเอกสารจริงก่อนส่งงาน
- เมื่อรีเซ็ต Runtime ข้อมูลใน `/content/` อาจหาย ให้เก็บ `best.pt` ไว้ใน Drive

## 0. ติดตั้งและตั้งค่า

In [7]:
%pip -q install ultralytics open_clip_torch gdown kagglehub chromadb sentence-transformers google-genai gradio

In [8]:
import os, glob, json, random, shutil, time
from pathlib import Path
from collections import Counter
import cv2, torch, numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO, YOLOWorld

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMGSZ = 224
device = DEVICE  # ชื่อตัวแปรที่โค้ด CLIP ต้นฉบับของเพื่อนใช้
BASE_DIR = Path('/content/trash_ai')
BASE_DIR.mkdir(parents=True, exist_ok=True)

# หากต้องการใช้โมเดลจาก Drive เปิดเป็น True ก่อน Run all
MOUNT_DRIVE = False
DRIVE_MODEL = Path('/content/drive/MyDrive/trash_ai/best.pt')
SAVE_TRAINED_MODEL_TO_DRIVE = False
TRAIN_IF_MISSING = True

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

model_candidates = [Path('/content/best.pt')]
if MOUNT_DRIVE:
    model_candidates.insert(0, DRIVE_MODEL)
WEIGHTS_PATH = next((p for p in model_candidates if p.is_file()), None)
print('Device:', DEVICE, '| Weights:', WEIGHTS_PATH or 'Not found — will train if allowed')

Device: cuda | Weights: Not found — will train if allowed


## 1. Dataset + Train (ส่วนเดิมของเพื่อน; รันเฉพาะเมื่อไม่มี best.pt)

In [9]:
if WEIGHTS_PATH is None:
    if not TRAIN_IF_MISSING:
        raise FileNotFoundError('ไม่พบ best.pt: อัปโหลดไว้ที่ /content/best.pt หรือเปิด MOUNT_DRIVE')
    import kagglehub
    dataset_root = kagglehub.dataset_download('sumn2u/garbage-classification-v2')
    print('Downloaded dataset:', dataset_root)

    IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    def has_images(folder):
        return any(f.is_file() and f.suffix.lower() in IMG_EXT for f in Path(folder).iterdir())

    class_root, best_n = None, 0
    for dp, dns, _ in os.walk(dataset_root):
        subs = [d for d in dns if has_images(Path(dp, d))]
        if len(subs) >= 2 and len(subs) == len(dns) and len(subs) > best_n:
            class_root, best_n = Path(dp), len(subs)
    if class_root is None:
        raise RuntimeError('หาโฟลเดอร์คลาสใน Dataset ไม่เจอ โปรดตรวจสอบโครงสร้างข้อมูล')
    classes = sorted(p.name for p in class_root.iterdir() if p.is_dir())
    print('Classes:', classes)

    random.seed(42)
    OUT = Path('/content/trash_data')
    if OUT.exists(): shutil.rmtree(OUT)
    for category in classes:
        examples = [f for f in (class_root/category).iterdir() if f.suffix.lower() in IMG_EXT]
        random.shuffle(examples)
        n = len(examples); n_train, n_val = int(n*0.8), int(n*0.1)
        split_files = {
            'train': examples[:n_train],
            'val': examples[n_train:n_train+n_val],
            'test': examples[n_train+n_val:]
        }
        for split, files_here in split_files.items():
            dest = OUT/split/category; dest.mkdir(parents=True, exist_ok=True)
            for file in files_here: shutil.copy2(file, dest/file.name)
    print('Prepared data at', OUT)

    # จาก Notebook เพื่อน: YOLO11s classification, 20 epochs, image size 224
    train_model = YOLO('yolo11s-cls.pt')
    train_model.train(data=str(OUT), epochs=5, imgsz=224, batch=32, patience=5)
    WEIGHTS_PATH = Path(train_model.trainer.save_dir)/'weights'/'best.pt'
    if not WEIGHTS_PATH.is_file(): raise FileNotFoundError('Training จบแล้ว แต่ไม่พบ best.pt')

    if SAVE_TRAINED_MODEL_TO_DRIVE and MOUNT_DRIVE:
        DRIVE_MODEL.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(WEIGHTS_PATH, DRIVE_MODEL)
        print('Saved to', DRIVE_MODEL)
else:
    print('ใช้ weights ที่มีอยู่ ไม่ฝึกใหม่:', WEIGHTS_PATH)

classifier = YOLO(str(WEIGHTS_PATH))
class_names = [classifier.names[i] for i in range(len(classifier.names))]
print('Loaded classifier:', WEIGHTS_PATH)
print('Classes:', class_names)

Using Colab cache for faster access to the 'garbage-classification-v2' dataset.
Downloaded dataset: /kaggle/input/garbage-classification-v2
Classes: ['battery', 'biological', 'cardboard', 'clothes', 'glass', 'metal', 'paper', 'plastic', 'shoes', 'trash']
Prepared data at /content/trash_data
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/trash_data, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None,

### ประเมินโมเดล (ถ้าฝึกใหม่ใน Runtime นี้)

In [10]:
if Path('/content/trash_data/test').is_dir():
    metrics = classifier.val(data='/content/trash_data', split='test')
    print('Top-1:', metrics.top1, '| Top-5:', metrics.top5)
else:
    print('ข้ามการประเมิน: ไม่มีชุด test ใน Runtime นี้ (แต่ยังใช้ best.pt inference ได้)')

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s-cls summary (fused): 47 layers, 5,446,938 parameters, 0 gradients, 12.1 GFLOPs
train: /content/trash_data/train... found 9802 images in 10/10 classes ✅ 
val: /content/trash_data/val... found 1221 images in 10/10 classes ✅ 
test: /content/trash_data/test... found 1236 images in 10/10 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 539.0±366.9 MB/s, size: 33.4 KB)
test: Scanning /content/trash_data/test... 1236 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1236/1236 2.7Kit/s 0.5s
test: New cache created: /content/trash_data/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 78/78 9.3it/s 8.4s
                   all      0.946      0.998
Speed: 0.1ms preprocess, 1.8ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val-2
Top-1: 0.9457928538322449 | Top-5: 0.9983818531036377


## 2. YOLO-World + CLIP + ฟังก์ชัน detect_trash() ของเพื่อน

In [6]:
import open_clip

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-16", pretrained="openai")
clip_model = clip_model.to(device).eval()
clip_tok = open_clip.get_tokenizer("ViT-B-16")

CLASS_PROMPTS = {
    "battery":    ["a battery", "used batteries", "a rechargeable battery"],
    "biological": ["food waste", "a banana peel", "an apple core", "fruit and vegetable scraps"],
    "cardboard":  ["a cardboard box", "corrugated cardboard", "flattened cardboard"],
    "clothes":    ["clothes", "a t-shirt", "a piece of fabric clothing"],
    "glass":      ["a glass bottle", "a glass jar", "broken glass"],
    "metal":      ["a metal can", "an aluminum soda can", "a tin can", "scrap metal"],
    "paper":      ["a sheet of paper", "crumpled paper", "a newspaper", "a disposable paper cup"],
    "plastic":    ["a plastic bottle", "a crushed plastic bottle", "a plastic bag", "plastic packaging", "a plastic cup"],
    "shoes":      ["a pair of shoes", "a sneaker", "a boot"],
    "trash":      ["general waste", "a black garbage bag", "styrofoam", "mixed rubbish"],
}

class_names = [classifier.names[i] for i in range(len(classifier.names))]
with torch.no_grad():
    rows = []
    for n in class_names:
        texts = [f"a photo of {p}" for p in CLASS_PROMPTS.get(n, [n])]
        e = clip_model.encode_text(clip_tok(texts).to(device))
        e = e / e.norm(dim=-1, keepdim=True)
        e = e.mean(0)
        rows.append(e / e.norm())
    text_feats = torch.stack(rows)

@torch.no_grad()
def clip_probs(crops_bgr):
    ims = torch.stack([clip_preprocess(Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))) for c in crops_bgr]).to(device)
    f = clip_model.encode_image(ims)
    f = f / f.norm(dim=-1, keepdim=True)
    return (100.0 * f @ text_feats.T).softmax(dim=-1).cpu().numpy()

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:555: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [11]:
# --- Eye: open-vocabulary detector that finds trash-like objects ---
detector = YOLOWorld("yolov8s-worldv2.pt")

PROMPTS = [
    "trash", "garbage", "plastic bottle", "glass bottle", "metal can", "aluminum can",
    "cardboard box", "paper", "crumpled paper", "plastic bag", "plastic cup",
    "food waste", "banana peel", "battery", "shoe", "clothes", "styrofoam", "wrapper",
]
detector.set_classes(PROMPTS)

requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 37 packages in 1.28s
Prepared 1 package in 3.88s
Installed 1 package in 1ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@7ffa84b3bfa40c42ecc1c77147a855e69cb2dd40)

requirements: AutoUpdate success ✅ 5.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|███████████████████████████████████████| 338M/338M [00:09<00:00, 36.3MiB/s]
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


In [12]:
rng = np.random.default_rng(3)
COLORS = {i: tuple(int(x) for x in rng.integers(50, 255, 3)) for i in classifier.names}  # BGR

def detect_trash(img_path, det_conf=0.08, final_conf=0.40, w_yolo=0.5, iou=0.5,
                 pad=0.05, min_area=0.002, max_area=0.90, show=True, debug=True):
    """w_yolo: weight of your trained classifier (0..1). 1-w_yolo goes to CLIP."""
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError(f'เปิดภาพไม่ได้: {img_path}')
    H, W = img.shape[:2]

    det = detector.predict(img, conf=det_conf, iou=iou, agnostic_nms=True, verbose=False)[0]
    boxes, crops = [], []
    for x1, y1, x2, y2 in det.boxes.xyxy.cpu().numpy():
        bw, bh = x2 - x1, y2 - y1
        if not (min_area <= bw * bh / (W * H) <= max_area):
            continue
        cx1, cy1 = int(max(0, x1 - pad * bw)), int(max(0, y1 - pad * bh))
        cx2, cy2 = int(min(W, x2 + pad * bw)), int(min(H, y2 + pad * bh))
        crops.append(img[cy1:cy2, cx1:cx2])
        boxes.append((int(x1), int(y1), int(x2), int(y2)))
    if not crops:
        crops, boxes = [img], [(0, 0, W - 1, H - 1)]

    yp = np.stack([p.probs.data.cpu().numpy() for p in classifier.predict(crops, imgsz=IMGSZ, verbose=False)])
    cp = clip_probs(crops)
    fp = w_yolo * yp + (1 - w_yolo) * cp

    out = img.copy()
    t = max(2, int(round(min(H, W) / 250)))
    fs = max(0.5, min(H, W) / 900)
    results = []
    for box, y, c, f in zip(boxes, yp, cp, fp):
        idx = int(f.argmax()); conf = float(f[idx])
        sure = conf >= final_conf
        label = class_names[idx] if sure else f"unsure ({class_names[idx]})"
        color = COLORS[idx] if sure else (128, 128, 128)
        results.append((box, class_names[idx] if sure else "unsure", conf))
        if debug:
            print(f"box {box}: yolo={class_names[int(y.argmax())]} {y.max():.0%} | clip={class_names[int(c.argmax())]} {c.max():.0%} | final={label} {conf:.0%}")
        x1, y1, x2, y2 = box
        cv2.rectangle(out, (x1, y1), (x2, y2), color, t)
        text = f"{label} {conf:.0%}"
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, fs, 2)
        ty = max(th + 6, y1)
        cv2.rectangle(out, (x1, ty - th - 6), (x1 + tw + 6, ty), color, -1)
        cv2.putText(out, text, (x1 + 3, ty - 4), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), 2, cv2.LINE_AA)

    summary = Counter(r[1] for r in results)
    if show:
        plt.figure(figsize=(9, 9))
        plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title(f"{len(results)} item(s): " + ", ".join(f"{k} x{v}" for k, v in summary.items()))
        plt.show()
    return results, summary

### ลองตรวจจับรูป (เลือกทดสอบเอง ไม่บังคับอัปโหลดระหว่าง Run all)
รัน Cell ข้างล่าง **เฉพาะเวลาต้องการทดสอบ YOLO**

In [13]:
# ถ้าจะทดสอบ ให้เอา # หน้าบรรทัดออก แล้วรัน Cell นี้เอง
# from google.colab import files
# uploaded = files.upload()
# for filename in uploaded:
#     detect_trash(filename, show=True, debug=True)
print('ระบบตรวจจับพร้อม: detect_trash(image_path)')

ระบบตรวจจับพร้อม: detect_trash(image_path)


## 3. Knowledge Base — ตัวอย่าง 10 ประเภท (ส่วนใหม่ของเรา)

In [14]:
# ข้อมูลตัวอย่างของโครงการเพื่อสาธิต ไม่ใช่แหล่งอ้างอิงภายนอก
knowledge = [{'id': 'battery_001', 'category': 'battery', 'title': 'แบตเตอรี่และถ่านไฟฉาย', 'content': 'แยกแบตเตอรี่ใช้แล้วออกจากขยะทั่วไป ไม่แกะ ไม่เจาะ ไม่เผา และนำส่งจุดรับแบตเตอรี่หรือจุดรับขยะอันตรายที่รองรับ หากบวมหรือรั่วควรหลีกเลี่ยงการสัมผัสโดยตรงและขอคำแนะนำจากผู้รับกำจัด', 'source': 'project_demo_draft'}, {'id': 'biological_001', 'category': 'biological', 'title': 'เศษอาหารและขยะอินทรีย์', 'content': 'แยกเศษผัก ผลไม้ และเศษอาหารออกจากขยะรีไซเคิล บางชนิดทำปุ๋ยหมักได้เมื่อจัดการอย่างเหมาะสม ควรจัดเก็บเพื่อลดกลิ่นและสัตว์รบกวน', 'source': 'project_demo_draft'}, {'id': 'cardboard_001', 'category': 'cardboard', 'title': 'กล่องกระดาษลัง', 'content': 'กล่องกระดาษลังที่แห้งและสะอาดโดยทั่วไปนำกลับรีไซเคิลได้ พับให้แบนก่อนส่ง และแยกส่วนที่เปื้อนอาหารหรือน้ำมันออก', 'source': 'project_demo_draft'}, {'id': 'clothes_001', 'category': 'clothes', 'title': 'เสื้อผ้าและสิ่งทอ', 'content': 'เสื้อผ้าที่สภาพดีสามารถใช้ซ้ำหรือบริจาคได้ เสื้อผ้าที่ชำรุดควรตรวจว่ามีโครงการรับคืนหรือรีไซเคิลสิ่งทอในพื้นที่หรือไม่', 'source': 'project_demo_draft'}, {'id': 'glass_001', 'category': 'glass', 'title': 'ขวดแก้วและแก้วแตก', 'content': 'ขวดแก้วบางชนิดส่งรีไซเคิลได้เมื่อจุดรับรองรับ ควรเทของเหลวออกก่อน แก้วแตกต้องเก็บในภาชนะหรือห่อที่แข็งแรงและติดป้ายเตือน เพื่อป้องกันบาดเจ็บ', 'source': 'project_demo_draft'}, {'id': 'metal_001', 'category': 'metal', 'title': 'กระป๋องและโลหะ', 'content': 'กระป๋องโลหะและอะลูมิเนียมหลายชนิดสามารถส่งรีไซเคิลได้ ควรเทสิ่งตกค้างออก ระวังขอบคม และตรวจเงื่อนไขจุดรับ', 'source': 'project_demo_draft'}, {'id': 'paper_001', 'category': 'paper', 'title': 'กระดาษ', 'content': 'กระดาษสะอาดและแห้งมักรีไซเคิลได้ ควรแยกกระดาษที่เปื้อนอาหาร น้ำมัน หรือสารเคมี และตรวจประเภทกระดาษที่จุดรับยอมรับ', 'source': 'project_demo_draft'}, {'id': 'plastic_001', 'category': 'plastic', 'title': 'พลาสติก', 'content': 'ขวด PET ที่ใส่น้ำดื่มโดยทั่วไปนำกลับรีไซเคิลได้ ควรเทของเหลวออกและคัดแยกตามชนิดพลาสติกและข้อกำหนดจุดรับ ไม่ใช่พลาสติกทุกประเภทจะรีไซเคิลได้เหมือนกัน', 'source': 'project_demo_draft'}, {'id': 'shoes_001', 'category': 'shoes', 'title': 'รองเท้า', 'content': 'รองเท้าที่ยังใช้งานได้ควรใช้ซ้ำหรือบริจาค รองเท้าเก่าที่ชำรุดตรวจหาจุดรับคืนหรือรีไซเคิลรองเท้าเฉพาะทางในพื้นที่', 'source': 'project_demo_draft'}, {'id': 'trash_001', 'category': 'trash', 'title': 'ขยะทั่วไป', 'content': 'ขยะทั่วไปที่ไม่อยู่ในกลุ่มรีไซเคิลหรือขยะอันตรายให้จัดการตามระบบเก็บขนของท้องถิ่น ห้ามปะปนแบตเตอรี่และขยะอันตรายลงถังขยะทั่วไป', 'source': 'project_demo_draft'}]

DATA_DIR = BASE_DIR/'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
with open(DATA_DIR/'waste_knowledge.json','w',encoding='utf-8') as f:
    json.dump(knowledge,f,ensure_ascii=False,indent=2)
print('Knowledge entries:', len(knowledge))

Knowledge entries: 10


## 4. Embeddings + ChromaDB (RAG Retrieval)

In [15]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device='cpu')
chroma_client = chromadb.PersistentClient(path=str(BASE_DIR/'chroma_db'))
collection = chroma_client.get_or_create_collection(
    name='trash_knowledge_v1', metadata={'hnsw:space':'cosine'}
)

# สำหรับรายการสั้น ๆ 1 ประเภท = 1 chunk; เมื่อมี PDF ยาวค่อยแบ่ง chunk เพิ่ม
ids = [item['id'] for item in knowledge]
documents = [item['title']+'\n'+item['content'] for item in knowledge]
metadatas = [{'category':item['category'],'title':item['title'],'source':item['source']} for item in knowledge]
vectors = embedding_model.encode(documents,normalize_embeddings=True).tolist()
old_ids = collection.get(include=[])['ids']
if old_ids: collection.delete(ids=old_ids)
collection.upsert(ids=ids,documents=documents,metadatas=metadatas,embeddings=vectors)
print('Vector DB entries:', collection.count())

KNOWN_CLASSES = set(item['category'] for item in knowledge)
def retrieve(question, category=None, top_k=3):
    if category is not None and category not in KNOWN_CLASSES:
        return []
    if collection.count() == 0: return []
    count = sum(1 for item in knowledge if item['category']==category) if category else collection.count()
    if count == 0: return []
    query_embedding = embedding_model.encode([question],normalize_embeddings=True).tolist()
    kwargs = dict(query_embeddings=query_embedding, n_results=min(top_k,count),
                  include=['documents','metadatas','distances'])
    if category: kwargs['where']={'category':category}
    res = collection.query(**kwargs)
    return [dict(content=d,category=m['category'],title=m['title'],source=m['source'],distance=float(dis))
            for d,m,dis in zip(res['documents'][0],res['metadatas'][0],res['distances'][0])]

print('Retrieval test:',[(x['category'],round(x['distance'],3)) for x in retrieve('ขวดพลาสติกทิ้งอย่างไร')])

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector DB entries: 10
Retrieval test: [('glass', 0.264), ('plastic', 0.319), ('metal', 0.441)]


## 5. เชื่อม Gemini (LLM) — ไม่ใส่ API Key ลงใน Notebook

In [16]:
from google import genai
from google.genai import types
from google.colab import userdata

# ใส่ GEMINI_API_KEY ในแท็บ Secrets (ไอคอนกุญแจ) และเปิด Notebook access
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = None

llm_client = genai.Client(api_key=API_KEY) if API_KEY else None
# เลือกโมเดลจากรายการที่ API Key ของคุณใช้งานได้; เปลี่ยนได้หาก 404/503
PREFERRED_MODELS = ['gemini-3.5-flash-lite','gemini-3.1-flash-lite','gemini-2.5-flash-lite']
AVAILABLE_MODELS=[]
if llm_client is not None:
    try:
        AVAILABLE_MODELS = [m.name.replace('models/','') for m in llm_client.models.list()
                            if 'generateContent' in (getattr(m,'supported_actions',None) or [])]
    except Exception as exc:
        print('เรียกรายชื่อโมเดลไม่ได้:',str(exc)[:250])
MODEL_NAME = next((m for m in PREFERRED_MODELS if m in AVAILABLE_MODELS), None)
if not MODEL_NAME and AVAILABLE_MODELS:
    MODEL_NAME = next((m for m in AVAILABLE_MODELS if 'flash' in m.lower()), AVAILABLE_MODELS[0])
print('Gemini model:', MODEL_NAME or 'ยังเลือกโมเดลไม่ได้')
if not API_KEY:
    print('ยังไม่มี API Key: เพิ่ม GEMINI_API_KEY ใน Colab Secrets แล้วรัน Cell นี้ใหม่')

Gemini model: ยังเลือกโมเดลไม่ได้
ยังไม่มี API Key: เพิ่ม GEMINI_API_KEY ใน Colab Secrets แล้วรัน Cell นี้ใหม่


## 6. Chatbot + RAG + Memory + Grounding

In [17]:
SYSTEM_PROMPT = """คุณเป็นผู้ช่วยแยกขยะ ตอบเป็นภาษาไทยที่กระชับ สุภาพ และใช้งานได้จริง
อ้างอิงความรู้ที่ให้ใน CONTEXT เท่านั้นสำหรับข้อเท็จจริงเกี่ยวกับวิธีจัดการขยะ
ห้ามอ้างว่าได้เห็นภาพเอง: หากมีผลตรวจจับ เป็นข้อมูลจากโมเดล YOLO/CLIP ไม่ใช่การดูภาพของคุณ
ถ้า CONTEXT ไม่ครอบคลุม ต้องบอกตามตรงว่าฐานความรู้ยังไม่มีคำตอบ
ไม่สร้างชื่อเอกสาร ลิงก์ หน่วยงาน หรือข้อมูลเกี่ยวกับข้อกำหนดท้องถิ่นขึ้นเอง
หากผลตรวจจับเป็น unsure ให้ระบุว่าไม่แน่ใจ และขอให้ผู้ใช้ยืนยัน
คำตอบควรมีคำแนะนำและบอกว่าแหล่งข้อมูลเป็นฐานความรู้ตัวอย่างภายในโครงการ"""

# แยก memory รายเซสชัน แทนการใช้ global chat_history ร่วมกันทุกคน
def build_context(question, history, detections=None):
    history = history or []
    # ให้ถามต่อเนื่องได้: ดึงหัวข้อจากประวัติ แต่จำกัดไม่ให้ยาวเกินไป
    recent_user = [m.get('content','') for m in history if m.get('role')=='user' and isinstance(m.get('content'),str)][-2:]
    search_text = ' '.join(recent_user+[question])[-1000:]
    docs = []
    if detections:
        for category in sorted(set(detections)):
            docs.extend(retrieve('วิธีคัดแยกและจัดการ '+category,category=category,top_k=1))
    else:
        docs = retrieve(search_text,top_k=3)
        # ผลค้นหาที่ไกลเกินไปมักไม่เกี่ยวข้อง ควรถามเพิ่มแทนเดา
        docs = [d for d in docs if d['distance'] <= 0.75]
    unique=[]; keys=set()
    for d in docs:
        key=(d['category'],d['content'])
        if key not in keys: unique.append(d); keys.add(key)
    return unique

def answer_rag(question, history=None, detections=None):
    if llm_client is None or not MODEL_NAME:
        return 'ยังไม่ได้ตั้งค่า Gemini API Key หรือไม่พบโมเดลที่ใช้งานได้ กรุณาตั้ง GEMINI_API_KEY แล้วรัน Cell 5 ใหม่', []
    docs = build_context(question,history,detections)
    if not docs:
        return 'ฐานความรู้ตัวอย่างยังไม่มีข้อมูลที่เกี่ยวข้องเพียงพอ กรุณาระบุประเภทขยะให้ชัดเจนหรือเพิ่มเอกสารใน Knowledge Base', []
    context = '\n\n'.join(f"[{i+1}] ประเภท: {d['category']} | ชื่อ: {d['title']} | แหล่ง: {d['source']}\n{d['content']}"
                          for i,d in enumerate(docs))
    recent = [f"{m.get('role','')}: {str(m.get('content',''))[:350]}" for m in (history or [])[-6:]]
    prompt = f"""CHAT HISTORY (ใช้เพื่อเข้าใจบริบท ไม่ใช่หลักฐาน):\n{chr(10).join(recent)}

DETECTION RESULTS (ถ้ามี): {detections or 'ไม่มี'}

CONTEXT (หลักฐาน):\n{context}

QUESTION: {question}

ตอบจาก CONTEXT เท่านั้น ระบุประเภทขยะเมื่อทราบ และเมื่อไม่แน่ใจต้องยอมรับข้อจำกัด
"""
    try:
        result = llm_client.models.generate_content(
            model=MODEL_NAME,contents=prompt,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT)
        )
        answer = result.text or 'โมเดลไม่ได้ส่งข้อความตอบกลับ'
    except Exception as exc:
        return f'เรียก Gemini ไม่สำเร็จ: {type(exc).__name__}: {str(exc)[:350]}', docs
    titles = ', '.join(sorted(set(d['title'] for d in docs)))
    return answer+'\n\nแหล่งความรู้: ฐานข้อมูลตัวอย่างภายในโครงการ ('+titles+')',docs

# ทดสอบแบบข้อความ (ต้องตั้ง API Key ก่อน)
if llm_client and MODEL_NAME:
    print(answer_rag('ขวดพลาสติกรีไซเคิลได้ไหม?')[0][:800])
else:
    print('ข้ามทดสอบ Gemini: ยังไม่มี API Key/Model')

ข้ามทดสอบ Gemini: ยังไม่มี API Key/Model


## 7. Gradio UI — รูปภาพ + กล่องแชท + Chat Memory

In [ ]:
import gradio as gr

# แสดงกล่องตรวจจับด้วย PIL (ต้นฉบับ detect_trash วาดภาพด้วย matplotlib เมื่อ show=True)
def annotate_image(image_path, results):
    img = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    for box,category,confidence in results:
        x1,y1,x2,y2=map(int,box)
        draw.rectangle((x1,y1,x2,y2),outline='#00b894' if category!='unsure' else '#e17055',width=4)
        draw.text((x1+3,max(0,y1-17)),f'{category} {confidence:.0%}',fill='#00b894')
    return img

def chat_step(message, image_path, visible_chat, memory, latest_detections):
    visible_chat = list(visible_chat or [])
    memory = list(memory or [])
    latest_detections = list(latest_detections or [])
    question = (message or '').strip()
    annotated = None
    if not question and image_path is None:
        return visible_chat,memory,latest_detections,None,None,''
    if image_path is not None:
        try:
            image_results, summary = detect_trash(image_path,show=False,debug=False)
            annotated = annotate_image(image_path,image_results)
            latest_detections = [k for k in summary if k in KNOWN_CLASSES]
            counts = ', '.join(f'{k} {summary[k]} ชิ้น' for k in summary)
            if not question: question='ขยะในภาพนี้ควรแยกและจัดการอย่างไร?'
            visible_chat.append({'role':'user','content':f'📷 ส่งภาพ | {question}'})
            if not latest_detections:
                answer='โมเดลยังไม่มั่นใจว่าเป็นขยะประเภทใด โปรดลองถ่ายให้ชัดขึ้นหรือพิมพ์ชื่อประเภทขยะ'
                docs=[]
            else:
                query = f'ภาพตรวจพบ {counts} คำถาม: {question}'
                answer,docs=answer_rag(query,memory,latest_detections)
            memory.append({'role':'user','content':f'ผลตรวจจับภาพ: {counts} | {question}'})
        except Exception as exc:
            visible_chat.append({'role':'user','content':'📷 '+question})
            answer=f'ไม่สามารถวิเคราะห์ภาพได้: {type(exc).__name__}: {str(exc)[:250]}'
            memory.append({'role':'user','content':question})
            latest_detections=[]
    else:
        visible_chat.append({'role':'user','content':question})
        # คำถามต่อเนื่องอ้างอิงภาพก่อนหน้าได้; ถ้าถามหัวข้อใหม่จะค้นจากข้อความเอง
        query = question
        if latest_detections and any(s in question for s in ['มัน','อันนี้','ชิ้นนี้','ต้อง','แล้ว','ที่เห็น','ในภาพ']):
            query = question+' (ภาพก่อนหน้าพบ '+', '.join(latest_detections)+')'
            answer,docs = answer_rag(query,memory,latest_detections)
        else:
            answer,docs = answer_rag(query,memory,None)
        memory.append({'role':'user','content':question})
    memory.append({'role':'assistant','content':answer})
    visible_chat.append({'role':'assistant','content':answer})
    return visible_chat,memory,latest_detections,annotated,None,''

def clear_all():
    return [],[],[],None,None,''

with gr.Blocks(title='Trash AI Chatbot') as demo:
    gr.Markdown('# ♻️ Trash AI Chatbot\nอัปโหลดภาพขยะ หรือถามเรื่องการคัดแยกและรีไซเคิล (Knowledge Base ฉบับเดโม)')
    chatbox = gr.Chatbot(label="บทสนทนา", height=420)
    with gr.Row():
        upload = gr.Image(type='filepath',label='รูปขยะ (ไม่บังคับ)',sources=['upload'])
        preview = gr.Image(label='ผลตรวจจับกรอบวัตถุ',interactive=False)
    question = gr.Textbox(label='คำถาม',placeholder='เช่น ขวดพลาสติกที่ตรวจพบต้องทิ้งยังไง?')
    with gr.Row():
        submit = gr.Button('ส่งข้อความ / วิเคราะห์ภาพ',variant='primary')
        reset = gr.Button('เริ่มแชทใหม่')
    memory_state = gr.State([])
    class_state = gr.State([])
    submit.click(chat_step,
                 inputs=[question,upload,chatbox,memory_state,class_state],
                 outputs=[chatbox,memory_state,class_state,preview,upload,question])
    question.submit(chat_step,
                    inputs=[question,upload,chatbox,memory_state,class_state],
                    outputs=[chatbox,memory_state,class_state,preview,upload,question])
    reset.click(clear_all,outputs=[chatbox,memory_state,class_state,preview,upload,question])

demo.launch(share=True,debug=True,show_error=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dacf1b8efa291efe40.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 8. ทดสอบและส่งงาน
**Checklist:**
1. ยืนยันว่าโหลด `best.pt` ของโมเดลที่เทรนจริง (`WEIGHTS_PATH` ไม่ว่าง)
2. อัปโหลดภาพขยะ → ได้ภาพกรอบ Bounding Box พร้อมประเภท
3. ถามว่า “ขยะในภาพนี้ทิ้งอย่างไร” → ได้คำตอบจากฐานความรู้ประเภทนั้น
4. ถามต่อว่า “แล้วต้องล้างไหม” → ระบบใช้บริบทการคุย/รูปก่อนหน้า
5. ลองถามนอกฐานข้อมูล → ระบบไม่ควรแต่งแหล่งอ้างอิง
6. ตรวจข้อมูลคำแนะนำด้วยเอกสารจริงก่อนส่งงาน และทำ Evaluation กับชุดคำถามทดสอบ

**ข้อจำกัด:** คะแนน Accuracy จาก YOLO classifier ไม่ใช่ความแม่นยำทั้งระบบ; ชุดข้อมูลตัวอย่างนี้ยังไม่มีหลักฐานภายนอกให้ cite; Gradio share link อาจเข้าถึงได้โดยบุคคลที่มีลิงก์; และการรันครั้งใหม่อาจต้องฝึกหรือโหลด `best.pt` จาก Drive อีกครั้ง